# HALW — 70feat_cnn

Baseline: textdescriptives features (intent: 70) → 1D CNN.

Notebook is Colab-first. Sections below follow the project skeleton: **00 Setup → 01 Imports → 02 Load → 03 Preprocess → 04 Train → 05 Evaluate**.

---
# 00 — Setup
---

In [1]:
# Clone or update the repo, then install runtime deps.
import os, sys

REPO_URL = "https://github.com/HaseebKhanYT/HALW.git"
REPO_DIR = "/content/HALW"
BRANCH = "setup/scaffold-halw-package"  # set to a branch name to test a PR before merge

if not os.path.exists(REPO_DIR):
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch -q && git checkout -q {BRANCH} && git pull --ff-only -q

!pip install -q "numpy<2.0" pandas textdescriptives spacy scikit-learn tensorflow kagglehub tqdm
!python -m spacy download en_core_web_lg -q

fatal: could not read Username for 'https://github.com': No such device or address
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 933.7 kB/s eta 0:00:0000:0100:05
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Restart the runtime once after the install so `numpy<2.0` takes effect, then continue from cell 01. 🔄

In [ ]:
# Optional: force-restart the runtime so the freshly installed numpy is loaded.
import os
os.kill(os.getpid(), 9)

: 

: 

: 

---
# 01 — Imports
---

In [1]:
import os, sys

REPO_DIR = "/content/HALW"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import numpy as np
import pandas as pd

from halw import data, features, preprocess, train, evaluate

from google.colab import drive
drive.mount("/content/drive")

ModuleNotFoundError: No module named 'halw'

---
# 02 — Load Dataset
---

Cached extracted features live on Drive — re-extraction takes a long time, so we read from cache when present.

In [ ]:
FEATURE_CACHE = "/content/drive/MyDrive/HALW/cache/shanegrami_70feat.csv"

if os.path.exists(FEATURE_CACHE):
    print(f"✅ Loading cached features from {FEATURE_CACHE}")
    df_features = pd.read_csv(FEATURE_CACHE)
else:
    print("⚠️ No cache found — extracting features (slow).")
    df = data.load_shanegrami(n_per_class=50000)
    df_features = features.extract_textdescriptives(df)
    os.makedirs(os.path.dirname(FEATURE_CACHE), exist_ok=True)
    df_features.to_csv(FEATURE_CACHE, index=False)
    print(f"💾 Saved cache to {FEATURE_CACHE}")

print(f"df_features shape: {df_features.shape}")

---
# 03 — Preprocess
---

In [ ]:
df_clean = preprocess.drop_sparse_columns(df_features, min_non_null_fraction=0.5)
print(f"After dropping sparse columns: {df_clean.shape}")

X = df_clean.drop(columns=["label"])
y = df_clean["label"]

X_train, X_val, X_test, y_train, y_val, y_test = preprocess.split(X, y)
X_train, X_val, X_test, scaler = preprocess.scale_for_conv1d(X_train, X_val, X_test)

X_train, y_train = preprocess.drop_nan_rows(X_train, y_train)
X_val, y_val = preprocess.drop_nan_rows(X_val, y_val)
X_test, y_test = preprocess.drop_nan_rows(X_test, y_test)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

---
# 04 — Train
---

In [ ]:
n_features = X_train.shape[1]
model = train.build_cnn(n_features=n_features)
history = train.train(model, X_train, y_train, X_val, y_val, epochs=50, batch_size=128)

---
# 05 — Evaluate
---

In [ ]:
metrics = evaluate.evaluate(model, X_test, y_test)
print(evaluate.format_metrics(metrics))

results_path = os.path.join(REPO_DIR, "results/results.csv")
evaluate.log_run(
    notebook="70feat_cnn",
    dataset="shanegrami_ai_human",
    n_samples=len(X_train) + len(X_val) + len(X_test),
    feature_pipeline="textdescriptives_all",
    n_features=n_features,
    model="cnn_baseline",
    metrics=metrics,
    notes="",
    path=results_path,
)
print(f"\n📝 Logged run to {results_path}")